# RAG for Plastic Classification Research

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

# ensure tracing is ready
print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_TRACING_V2"))
print(os.getenv("LANGSMITH_PROJECT"))

None
true
Home RAG


In [3]:
import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [5]:
from langsmith import traceable

## Indexing

### Loading Documents 


In [6]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

data_folder = Path("../data")  # folder where your PDFs are stored
pdf_files = list(data_folder.glob("*.pdf"))  # all PDFs in the folder

docs = []

for pdf in pdf_files:
    loader = PyMuPDFLoader(str(pdf))
    docs.extend(loader.load())

print(f"Total documents loaded: {len(docs)}")
print(f"First doc preview:\n{docs[0].page_content[:500]}...")

C:\Users\dorky\AppData\Local\Temp\ipykernel_33176\2881700297.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Total documents loaded: 150
First doc preview:
Review
Microplastics in freshwaters and drinking water: Critical review and
assessment of data quality
Albert A. Koelmans a, *, Nur Hazimah Mohamed Nor a, Enya Hermsen a, Merel Kooi a,
Svenja M. Mintenig b, c, Jennifer De France d, **
a Aquatic Ecology and Water Quality Management Group, Wageningen University, the Netherlands
b Copernicus Institute of Sustainable Development, Utrecht University, the Netherlands
c KWR Watercycle Research Institute, Nieuwegein, the Netherlands
d World Health Organ...


### Splitting documents

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # adjust to fit model tokens
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

split_docs = text_splitter.split_documents(docs)
print(f"Total chunks for RAG: {len(split_docs)}")

Total chunks for RAG: 926


### Storing documents

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)

document_ids = vector_store.add_documents(documents=split_docs)
print(document_ids[:3])

['a6c67548-9e96-4f29-8a52-5a75e5672c82', '79072280-8b33-4fca-a073-e807b445e92c', '9a6df350-6644-4436-b479-d2bcc6172431']


## Retrieval and generation

In [ ]:
# from eval import retrieve_docs

In [19]:
# create a retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

In [20]:
def retrieve_docs(question):
    return retriever.invoke(question)

In [21]:
# format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [22]:
# create prompt template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
""")

In [23]:
# instantiate model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",  # or whatever model you want
    temperature=0
)

In [24]:
# connect retriever to LLM
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [25]:
def run_rag(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    answer = rag_chain.invoke(question)

    return {
        "question": question,
        "contexts": [d.page_content for d in docs],
        "answer": answer
    }

In [26]:
samples = [
    run_rag("What are microplastics doing to human health?"),
    run_rag("How do microplastics enter the ocean?"),
]

In [ ]:
from ragas import EvaluationDataset

dataset = EvaluationDataset.from_list(samples)

In [ ]:
# query
# rag_chain.invoke("What do the documents say about the effect of microplastics on humans?")

'The documents indicate that while adverse effects of microplastics are known in marine organisms, the health effects on humans are less well-studied. The discussion on health and diseases associated with microplastics is primarily based on in-vitro studies, small pilot human studies, and speculation from changes observed in marine organisms. Notable adverse effects identified include genotoxicity and cytotoxicity, with in-vitro studies showing that microplastics can increase the frequency of micronucleation and other nuclear abnormalities in human blood lymphocytes. These effects have been linked to disorders such as infertility, diabetes, obesity, and cardiovascular disease. Additionally, the size of microplastics is critical, as smaller particles (less than 100 μm) can penetrate biological barriers and accumulate in tissues, including the placenta. While microplastics can be ingested or inhaled, the human health effects remain largely unknown, and limited data suggest potential part

### Evaluate model

In [ ]:
# get top-k chunks
docs_with_scores = vector_store.similarity_search_with_score(
    "What are microplastics?",
    k=5
)

In [ ]:
evaluator = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def evaluate_answer(reference, prediction):
    prompt = f"""
    Reference Answer:
    {reference}

    Predicted Answer:
    {prediction}

    Score from 0 to 3:
    0 = Completely wrong
    1 = Mostly wrong
    2 = Mostly correct
    3 = Fully correct

    Only output the score.
    """
    return evaluator.predict(prompt)